# 🇻🇳 App Phân Loại Tin Tức Vietnamnet — Ensemble SVM + PhoBERT

| Cell | Nội dung |
|------|----------|
| **0. Kiểm tra** | Xác nhận thư viện và cả 2 mô hình |
| **1. Kiểm tra app** | Xác nhận `app_combined.py` tồn tại |
| **2. Khởi động** | Mở Streamlit tại `http://localhost:8501` |

> Sau khi train xong cả SVM lẫn PhoBERT, chạy **Cell 0 → Cell 2** là dùng được ngay.

In [1]:
# -- Kiem tra moi truong ------------------------------------------------
import os, sys

SVM_PIPELINE  = os.path.normpath(os.path.join(os.getcwd(), '..', 'SVM',     'model', 'inference_pipeline.pkl'))
PHO_MODEL_DIR = os.path.normpath(os.path.join(os.getcwd(), '..', 'PhoBERT', 'model'))
PHO_CONFIG    = os.path.join(PHO_MODEL_DIR, 'label_config.json')
PHO_MODEL     = os.path.join(PHO_MODEL_DIR, 'model.safetensors')
PHO_THR       = os.path.join(PHO_MODEL_DIR, 'thresholds.json')

print('Kiem tra thu vien:')
ok = True
missing_pkgs = []
for _pkg in ['streamlit', 'requests', 'bs4', 'pyvi', 'torch', 'transformers',
             'numpy', 'pandas', 'scipy', 'sklearn']:
    try:
        __import__(_pkg)
        print(f'  OK  {_pkg}')
    except ImportError:
        print(f'  MISS {_pkg}')
        missing_pkgs.append(_pkg)
        ok = False

print()
print('Kiem tra model SVM:')
if os.path.exists(SVM_PIPELINE):
    print(f'  OK  inference_pipeline.pkl  ({os.path.getsize(SVM_PIPELINE)/1e6:.1f} MB)')
else:
    print(f'  MISS {SVM_PIPELINE}')
    ok = False

print()
print('Kiem tra model PhoBERT:')
_required_pho = [
    ('model.safetensors',     PHO_MODEL),
    ('label_config.json',     PHO_CONFIG),
    ('config.json',           os.path.join(PHO_MODEL_DIR, 'config.json')),
    ('tokenizer_config.json', os.path.join(PHO_MODEL_DIR, 'tokenizer_config.json')),
]
for _name, _path in _required_pho:
    if os.path.exists(_path):
        print(f'  OK  {_name}  ({os.path.getsize(_path)/1e6:.1f} MB)')
    else:
        print(f'  MISS {_name}')
        ok = False

if os.path.exists(PHO_THR):
    print('  OK  thresholds.json -- Threshold calibration: BAT')
else:
    print('  WARN thresholds.json chua co -- app dung threshold mac dinh')

if not ok:
    print()
    print('=' * 60)
    print('  KHONG THE TIEP TUC')
    print('=' * 60)
    if not os.path.exists(SVM_PIPELINE):
        print('  SVM model chua duoc tao.')
        print('  -> Chay main_SVM.ipynb (Section 7) de export inference_pipeline.pkl.')
    if not os.path.exists(PHO_MODEL):
        print('  PhoBERT model chua duoc tao.')
        print('  -> Chay main_PhoBERT.ipynb (Section 7) de export model.')
    if missing_pkgs:
        print('  Thieu thu vien. Cai dat:')
        print(f'     pip install {chr(32).join(missing_pkgs)}')
    print('=' * 60)
    raise SystemExit('Chua san sang -- xem huong dan o tren.')

print()
print('San sang! Chay cell tiep theo de khoi dong app.')

Kiem tra thu vien:
  OK  streamlit
  OK  requests
  OK  bs4
  OK  pyvi
  OK  torch


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  OK  transformers
  OK  numpy
  OK  pandas
  OK  scipy
  OK  sklearn

Kiem tra model SVM:
  OK  inference_pipeline.pkl  (29.7 MB)

Kiem tra model PhoBERT:
  OK  model.safetensors  (540.1 MB)
  OK  label_config.json  (0.0 MB)
  OK  config.json  (0.0 MB)
  OK  tokenizer_config.json  (0.0 MB)
  OK  thresholds.json -- Threshold calibration: BAT

San sang! Chay cell tiep theo de khoi dong app.


---
## ✍️ Cell 1 — Kiểm tra App

Xác nhận `app_combined.py` tồn tại và các tính năng chính.

In [2]:
import os

_app_file = os.path.join(os.getcwd(), 'app_combined.py')

if os.path.exists(_app_file):
    _size = os.path.getsize(_app_file) / 1024
    print(f'✅ app_combined.py ton tai  ({_size:.1f} KB)')

    with open(_app_file, encoding='utf-8') as f:
        _src = f.read()
    if 'ensemble_predict' in _src:
        print('✅ Ensemble predict: co trong app')
    if 'get_top_keywords' in _src:
        print('✅ Top keywords: co trong app')
    if 'batch' in _src.lower():
        print('✅ Batch URL: co trong app')
else:
    print(f'❌ Khong tim thay: {_app_file}')
    print('   -> Kiem tra lai thu muc Combined_Model_App/')

print('-> Chay Cell 2 de khoi dong app.')

✅ app_combined.py ton tai  (27.7 KB)
✅ Ensemble predict: co trong app
✅ Top keywords: co trong app
✅ Batch URL: co trong app
-> Chay Cell 2 de khoi dong app.


---
## 🚀 Cell 2 — Khởi Động App

Mở **http://localhost:8501** trên trình duyệt.

> Cell này chạy cho đến khi bạn nhấn **■ Stop** (Interrupt Kernel).  
> Hoặc mở terminal riêng: `streamlit run app_combined.py`

In [ ]:
import subprocess, sys, os, threading, webbrowser, time

_cmd = [sys.executable, '-m', 'streamlit', 'run', 'app_combined.py',
        '--server.port', '8501', '--server.headless', 'true']
print('Chay:', ' '.join(_cmd))
print('Mo trinh duyet: http://localhost:8501')
print('Nhan ■ Interrupt Kernel de dung.')
print()

threading.Thread(
    target=lambda: (time.sleep(4), webbrowser.open('http://localhost:8501')),
    daemon=True
).start()

subprocess.run(_cmd, cwd=os.getcwd())

Chay: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m streamlit run app_combined.py --server.port 8501 --server.headless true
Mo trinh duyet: http://localhost:8501
Nhan ■ Interrupt Kernel de dung.

